# Risky-Debt Model — SMM Estimation

Structural estimation via Simulated Method of Moments.
Each optimizer evaluation triggers `solve_risky_debt` (nested VFI with z-interpolation + adaptive b' bounds).
Two-step: Stage 1 with W = I (global), Stage 2 with W = Omega_hat^{-1} (local).

In [1]:
import contextlib, io, logging, os, sys, time, warnings
from dataclasses import asdict
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
warnings.filterwarnings("ignore", message=r".*does not produce the same series.*")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display

from src.v2.estimation import SMMMonteCarloConfig, SMMRunConfig, solve_smm
from src.v2.evaluation import prepare_evaluation_run, save_manifest_sections, save_summary_rows
from src.v2.environments.risky_debt import EconomicParams, RiskyDebtEnv, ShockParams
from src.v2.solvers import RiskyDebtSolverConfig, solve_risky_debt
from src.v2.utils.seeding import fold_in_seed

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = "{:.4f}".format

def _run_quietly(fn, *args, **kwargs):
    buf = io.StringIO()
    t0 = time.perf_counter()
    with contextlib.redirect_stdout(buf):
        result = fn(*args, **kwargs)
    return result, time.perf_counter() - t0

# Short labels for display
_ML = {"avg_equity_issuance_assets": "Avg Iss/k", "conditional_issuance_size": "Cond Iss",
       "autocorr_equity_issuance": "AC Iss", "crosscorr_leverage_issuance": "Corr(Lev,Iss)",
       "avg_net_debt_assets": "Avg Lev", "std_leverage": "Std Lev",
       "serial_corr_investment": "AC I/k", "var_investment_assets": "Var I/k",
       "income_ar1_beta": "AR1 beta", "income_ar1_resid_std": "AR1 sigma",
       "default_frequency": "Def Freq"}
_PL = {"alpha": "alpha", "psi1": "psi1", "eta0": "eta0", "eta1": "eta1",
       "c_def": "c_def", "rho": "rho", "sigma_epsilon": "sigma"}

/Users/wangzhaoxuan/Desktop/JPM-TSRL/DL_corp_finance/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
# ── Profile ───────────────────────────────────────────────────────────
PROFILE = "FULL"  # "SMOKE" or "FULL"

_P = {
    "SMOKE": dict(solver=dict(n_k=15, n_b=20, n_z=10, n_z_solve=5),
                  smm=dict(n_firms=300, horizon=25, burn_in=25, n_sim_panels=30,
                           global_method="differential_evolution", local_method="Powell",
                           optimizer_maxiter=5, optimizer_popsize=5, master_seed=(20, 26)),
                  mc=dict(n_replications=2)),
    "FULL": dict(solver=dict(n_k=25, n_b=50, n_z=25, n_z_solve=10),
                 smm=dict(n_firms=5000, horizon=25, burn_in=75, n_sim_panels=50,
                          global_method="differential_evolution", local_method="Powell",
                          optimizer_maxiter=50, master_seed=(20, 26)),
                 mc=dict(n_replications=5)),
}

# ── Solver: nested VFI with z-interpolation + adaptive b' bounds ──────
# Set adaptive=False to disable boundary discovery (full b' range).
ADAPTIVE     = True
BUFFER_FRAC  = 0.10   # buffer around discovered active b' region

# ── Structural parameters ─────────────────────────────────────────────
ESTIMATED_PARAMS = ("alpha", "psi1", "eta0", "eta1", "c_def", "rho", "sigma_epsilon")

TRUE_PARAMS = dict(production_elasticity=0.70, cost_convex=0.05,
                   cost_inject_fixed=0.10, cost_inject_linear=0.05,
                   default_haircut=0.45, rho=0.60, sigma=0.15)

CALIBRATED = dict(interest_rate=0.05, depreciation_rate=0.15, tax=0.30)

BETA_BOUNDS = ((0.30,0.90), (0.01,0.25), (0.01,0.30), (0.01,0.20),
               (0.20,1.0), (0.30,0.90), (0.05,0.50))
BETA_INITIAL_GUESS = [0.5, 0.13, 0.15, 0.10, 0.7, 0.55, 0.25]

# ── Build objects ─────────────────────────────────────────────────────
env = RiskyDebtEnv(
    econ_params=EconomicParams(
        interest_rate=CALIBRATED["interest_rate"],
        depreciation_rate=CALIBRATED["depreciation_rate"],
        tax=CALIBRATED["tax"],
        production_elasticity=TRUE_PARAMS["production_elasticity"],
        cost_convex=TRUE_PARAMS["cost_convex"],
        default_haircut=TRUE_PARAMS["default_haircut"],
        cost_inject_fixed=TRUE_PARAMS["cost_inject_fixed"],
        cost_inject_linear=TRUE_PARAMS["cost_inject_linear"],
    ),
    shock_params=ShockParams(mu=0.0, rho=TRUE_PARAMS["rho"], sigma=TRUE_PARAMS["sigma"]),
    k_min_mult=0.05, k_max_mult=1.5, b_max_mult=4.0, b_min_mult=0.2,
)

p = _P[PROFILE]
SOLVER_CONFIG = RiskyDebtSolverConfig(
    **p["solver"],
    adaptive=ADAPTIVE,
    buffer_frac=BUFFER_FRAC,
)
SMM_RUN_CONFIG = SMMRunConfig(**p["smm"])

RUN = prepare_evaluation_run("smm_risky_debt", save_run=True,
                             run_tag="smm-risky-debt-demo",
                             results_root=str(REPO_ROOT / "outputs" / "notebooks"))
save_manifest_sections(RUN, profile=PROFILE, estimated=list(ESTIMATED_PARAMS),
                       calibrated=CALIBRATED, true_params=TRUE_PARAMS,
                       solver=asdict(SOLVER_CONFIG), smm=asdict(SMM_RUN_CONFIG))

print(f"Profile: {PROFILE}")
print(f"Solver:  k={SOLVER_CONFIG.n_k} x b={SOLVER_CONFIG.n_b} x z={SOLVER_CONFIG.n_z} "
      f"(z_solve={SOLVER_CONFIG.n_z_solve})")
print(f"         adaptive={ADAPTIVE}, buffer={BUFFER_FRAC:.0%}")
print(f"SMM:     firms={SMM_RUN_CONFIG.n_firms}, T={SMM_RUN_CONFIG.horizon}, "
      f"S={SMM_RUN_CONFIG.n_sim_panels}, maxiter={SMM_RUN_CONFIG.optimizer_maxiter}")

Profile: FULL
Solver:  k=25 x b=50 x z=25 (z_solve=10)
         adaptive=True, buffer=10%
SMM:     firms=5000, T=25, S=50, maxiter=50


In [3]:
beta_true = env.smm_true_beta(estimated_params=ESTIMATED_PARAMS)
spec = env.make_smm_spec(solver_config=SOLVER_CONFIG, initial_guess=BETA_INITIAL_GUESS,
                         bounds=BETA_BOUNDS, estimated_params=ESTIMATED_PARAMS)

display(pd.DataFrame({
    "param": [_PL[p] for p in spec.parameter_names],
    "true": beta_true, "init": spec.initial_guess,
    "lo": [b[0] for b in spec.bounds], "hi": [b[1] for b in spec.bounds],
}))
print(f"Moments ({len(spec.moment_names)}): {[_ML[m] for m in spec.moment_names]}")

# Reference solve + target moments
solved, t = _run_quietly(solve_risky_debt, env, config=SOLVER_CONFIG)
print(f"\nRef solve: {solved['stop_reason']}, outer={solved['n_outer']}, wall={t:.1f}s")
if "adaptive_b_bounds" in solved:
    print(f"Adaptive b': [{solved['adaptive_b_bounds'][0]:.1f}, {solved['adaptive_b_bounds'][1]:.1f}]")

target_seed = fold_in_seed(SMM_RUN_CONFIG.master_seed, "smm", "risky_debt", "target")
target, t = _run_quietly(spec.simulate_target_moments, beta_true, SMM_RUN_CONFIG, target_seed)
print(f"Target moments: generated in {t:.1f}s")

,param,true,init,lo,hi
0,alpha,0.7000,0.5000,0.3000,0.9000
1,psi1,0.0500,0.1300,0.0100,0.2500
2,eta0,0.1000,0.1500,0.0100,0.3000
3,eta1,0.0500,0.1000,0.0100,0.2000
4,c_def,0.4500,0.7000,0.2000,1.0000
5,rho,0.6000,0.5500,0.3000,0.9000
6,sigma,0.1500,0.2500,0.0500,0.5000


Moments (11): ['Avg Iss/k', 'Cond Iss', 'AC Iss', 'Corr(Lev,Iss)', 'Avg Lev', 'Std Lev', 'AC I/k', 'Var I/k', 'AR1 beta', 'AR1 sigma', 'Def Freq']

Ref solve: converged_outer_value, outer=8, wall=32.9s
Adaptive b': [56.8, 94.9]
Target moments: generated in 33.4s


## Diagnostics: Oracle + Jacobian

In [ ]:
# Oracle test: Q(beta_true) ≈ 0
oracle_seed = fold_in_seed(SMM_RUN_CONFIG.master_seed, "smm", "oracle")
oracle_result, t = _run_quietly(spec.simulate_panel_moments, beta_true, SMM_RUN_CONFIG, oracle_seed)
oracle_err = oracle_result.panel_moments.mean(axis=0) - target.values
Q = float(oracle_err @ oracle_err)
print(f"Oracle: Q={Q:.2e}, ||e||={np.sqrt(Q):.2e}, wall={t:.1f}s — {'PASS' if Q < 0.01 else 'FAIL'}")

# Jacobian (finite differences)
print(f"\nJacobian: {2*len(beta_true)} solves ...")
fd_steps = 0.01 * (np.array(BETA_BOUNDS)[:, 1] - np.array(BETA_BOUNDS)[:, 0])
K, R = len(beta_true), len(target.values)
jac = np.zeros((R, K))

t0 = time.perf_counter()
for j in range(K):
    bp, bm = beta_true.copy(), beta_true.copy()
    bp[j] += fd_steps[j]; bm[j] -= fd_steps[j]
    mp, _ = _run_quietly(spec.simulate_panel_moments, bp, SMM_RUN_CONFIG, oracle_seed)
    mm, _ = _run_quietly(spec.simulate_panel_moments, bm, SMM_RUN_CONFIG, oracle_seed)
    jac[:, j] = (mp.panel_moments.mean(0) - mm.panel_moments.mean(0)) / (2 * fd_steps[j])

sv = np.linalg.svd(jac, compute_uv=False)
rank = int(np.sum(sv > 1e-8))
print(f"  SV: {sv}")
print(f"  Rank: {rank}/{K} — wall={time.perf_counter()-t0:.1f}s")

# Sensitivity table
ml = [_ML[m] for m in spec.moment_names]
pl = [_PL[p] for p in spec.parameter_names]
norms = np.linalg.norm(jac, axis=0)
status = ["DEAD" if n < 1e-6 else "WEAK" if n < 0.05 else "OK" for n in norms]
display(pd.DataFrame({"param": pl, "||D||": norms, "status": status}))
display(pd.DataFrame(jac, index=ml, columns=pl).round(4))

dead = [p for p, s in zip(pl, status) if s == "DEAD"]
weak = [p for p, s in zip(pl, status) if s == "WEAK"]
if dead: print(f"Dead: {dead}")
if weak: print(f"Weak: {weak}")

Oracle: Q=1.23e-05, ||e||=3.50e-03, wall=36.9s — PASS

Jacobian: 14 solves ...
  SV: [11.6857  5.1931  3.052   1.2093  0.2607  0.1093  0.0003]
  Rank: 7/7 — wall=516.8s


,param,||D||,status
0,alpha,4.0542,OK
1,psi1,4.0964,OK
2,eta0,3.6755,OK
3,eta1,6.7756,OK
4,c_def,0.0031,WEAK
5,rho,2.2132,OK
6,sigma,8.7663,OK


,alpha,psi1,eta0,eta1,c_def,rho,sigma
Avg Iss/k,-0.0848,0.1552,-0.1759,-0.4119,0.0001,0.0918,-0.0727
Cond Iss,-2.5191,-1.5017,0.3775,0.1438,0.0006,-0.6805,-0.8701
AC Iss,2.7251,1.5277,2.2022,4.5305,0.0026,0.7936,7.5182
"Corr(Lev,Iss)",0.3200,-1.9772,2.7522,4.9054,0.0015,-0.7789,4.1053
Avg Lev,0.4743,0.5772,-0.3644,-0.6817,0.0002,0.1285,-0.3834
Std Lev,-0.0445,-0.0972,0.1503,0.2861,0.0000,0.0116,0.2866
AC I/k,-1.0688,2.0416,-0.8099,-0.7397,0.0000,1.3487,-0.3162
Var I/k,0.1796,-0.1248,-0.0160,-0.0038,0.0000,0.0666,0.3827
AR1 beta,-1.0705,1.9281,-0.3139,0.1927,0.0000,1.1613,0.3516
AR1 sigma,-0.0869,0.1119,-0.0349,-0.0020,0.0000,0.0385,1.4522


Weak: ['c_def']


## Two-Step SMM Estimation

In [5]:
smm_result, wall = _run_quietly(solve_smm, spec, target, SMM_RUN_CONFIG)

# Table 1: Moment fit
moment_fit = pd.DataFrame({
    "moment": [_ML[m] for m in spec.moment_names],
    "target": smm_result.target_moments,
    "fitted": smm_result.stage2.average_moments,
})
print("Moment Fit"); display(moment_fit)

# Table 2: Parameter estimates
beta_hat = smm_result.beta_hat
se = smm_result.standard_errors
t_stat = (beta_hat - beta_true) / np.where(se > 0, se, np.nan)

param_est = pd.DataFrame({
    "param": [_PL[p] for p in spec.parameter_names],
    "true": beta_true, "est": beta_hat, "se": se, "t": t_stat,
    "p": 2 * (1 - norm.cdf(np.abs(t_stat))),
})
print("\nParameter Estimates"); display(param_est)

print(f"\nJ={smm_result.j_statistic:.4f}, p={smm_result.j_p_value:.4f}, df={smm_result.j_df}")
print(f"Stage 1: {SMM_RUN_CONFIG.global_method} nfev={smm_result.stage1.optimizer_nfev}")
print(f"Stage 2: Powell nfev={smm_result.stage2.optimizer_nfev}")
print(f"Wall: {wall:.1f}s")

save_summary_rows(RUN, moment_fit.to_dict("records"), filename="moment_fit.csv")
save_summary_rows(RUN, param_est.to_dict("records"), filename="parameter_estimates.csv")

/var/folders/8m/_52z3gvn0p58rgf1n5r1tw5h0000gn/T/ipykernel_4027/2983863396.py:33: RuntimeWarning: Omega_hat condition number 2.82e+09 exceeds 1e+06. W = Omega_hat^{-1} is numerically unstable. Falling back to W = I for Stage 2 (first-step estimator). J-test and efficient standard errors are not available. Increase n_sim_panels (recommended S >> R, e.g., 50x n_moments) to obtain the efficient estimator.
  result = fn(*args, **kwargs)


Moment Fit


,moment,target,fitted
0,Avg Iss/k,0.0168,0.0235
1,Cond Iss,0.0881,0.1917
2,AC Iss,-0.0507,-0.0518
3,"Corr(Lev,Iss)",0.2723,0.3008
4,Avg Lev,0.9099,0.8635
5,Std Lev,0.0308,0.0485
6,AC I/k,0.0023,0.0023
7,Var I/k,0.0287,0.0395
8,AR1 beta,0.3399,0.3639
9,AR1 sigma,0.2158,0.3139



Parameter Estimates


,param,true,est,se,t,p
0,alpha,0.7000,0.6213,0.0174,-4.5287,0.0000
1,psi1,0.0500,0.0691,0.0071,2.6860,0.0072
2,eta0,0.1000,0.1207,0.0418,0.4958,0.6200
3,eta1,0.0500,0.0851,0.0049,7.1274,0.0000
4,c_def,0.4500,0.3844,0.3896,-0.1682,0.8664
5,rho,0.6000,0.6411,0.0624,0.6578,0.5106
6,sigma,0.1500,0.2206,0.0306,2.3066,0.0211



J=nan, p=nan, df=4
Stage 1: differential_evolution nfev=5483
Stage 2: Powell nfev=369
Wall: 175418.4s


PosixPath('/Users/wangzhaoxuan/Desktop/JPM-TSRL/DL_corp_finance/outputs/notebooks/smm_risky_debt/smm-risky-debt/parameter_estimates.csv')